<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Choose a split

**[Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/)** · SQL and Data Preparation for AI · Module 5, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare random, time-based and customer-based splits, choose the one that matches how the model will be used, and write the split and its manifest with `split.py`.

| | |
|---|---|
| **Time** | About 50 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6. |
| **You should know** | The modelling table and its target, from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/). |
| **You do not need** | The local project from Modules 1 to 5. The setup below builds the database and the modelling table here. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every script, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/) has the full explanations, the interactive split explorer and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M05-L02-choose-a-split/choose-a-split-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/), the clean tables `data/clean/` from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/), the modelling table `data/model/tickets_model.csv` from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `build_features.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract", "clean", "build_features"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Why split at all

A model is useful only on tickets it has not seen. So you keep some tickets away from it, and test it on them later. A **split** divides the rows of the modelling table into parts with different jobs:

| Part | Job | Rule |
|---|---|---|
| **Train set** | The model learns from these rows. | Everything that learns from data uses only these rows. |
| **Validation set** | You compare versions and choose settings. | You look at its score many times. |
| **Test set** | One final, honest score. | You look at its score once, at the end. |

The question of this lesson is **which** rows go into each part.

## 2. Three ways to split, on eight tickets

The next cell makes eight tickets from three customers, A, B and C, in order of time. Then it splits them in three ways. Each split puts six tickets in train and two in test:

- **random**: two tickets chosen by chance (`sample` with a fixed `random_state`, so the draw is the same each time).
- **by_time**: the tickets from 1 May on.
- **by_customer** (a **group split**): all tickets of customer B.

> **Predict.** Which split puts the June ticket, T-8, in train? Then run the cell.

In [ ]:
import json
import pandas as pd

tiny = pd.DataFrame({
    "ticket": ["T-1", "T-2", "T-3", "T-4", "T-5", "T-6", "T-7", "T-8"],
    "customer": ["A", "B", "A", "C", "B", "A", "C", "A"],
    "created": pd.to_datetime(["2026-01-10", "2026-02-02", "2026-02-20", "2026-03-15",
                               "2026-04-05", "2026-04-30", "2026-05-20", "2026-06-12"]),
})
random_test = tiny.sample(n=2, random_state=42).index
tiny["random"] = "train"
tiny.loc[random_test, "random"] = "test"
tiny["by_time"] = "train"
tiny.loc[tiny["created"] >= "2026-05-01", "by_time"] = "test"
tiny["by_customer"] = "train"
tiny.loc[tiny["customer"] == "B", "by_customer"] = "test"
tiny

> **Check.** You should see `test` for T-2 and T-6 in `random`, for T-7 and T-8 in `by_time`, and for T-2 and T-5 in `by_customer`.

- **Random:** T-7 and T-8, the newest tickets, are in train. The model learns from May and June, then is tested on February and April.
- **By time:** the model learns from the past and is tested on the future, as in real use.
- **By customer:** customer B is only in test. The test asks: how well does the model work for a customer it never saw?

None of these is always right. The correct split copies the way the model is used. Ask three questions:

1. **Will the model predict the future from the past?** Then split by time.
2. **Will the model meet new customers, patients, shops or machines?** Then keep each group in one part.
3. **Are the rows independent, with no time order and no repeated groups?** Only then is a random split a fair test.

The [split explorer on the lesson page](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/#try-the-three-splits) shows the three splits on 60 tickets. Try it now, then come back.

## 3. The real data: something changed in May

Mei counts the tickets and the escalations per month. Run the cell.

In [ ]:
model = pd.read_csv("data/model/tickets_model.csv", parse_dates=["created_at"])
month = model["created_at"].dt.strftime("%Y-%m")
print(model.groupby(month)["escalated_72h"].agg(["size", "sum", "mean"]).round(3))

> **Check.** You should see six months. From January to April, the rate `mean` is between 0.110 and 0.129. In May, it is **0.198**. June is 0.163.

Grace, the support lead, knows why: "On 1 May, a new warranty policy started. Warranty questions now go to a senior agent more often." The world after training is not the world of the training data.

## 4. A random split hides the change

Now make a random 20% test set with pandas only.

> **Predict.** Will the test rate be close to the train rate, or higher? Then run the cell.

In [ ]:
test = model.sample(frac=0.2, random_state=0)
train = model.drop(test.index)
print(len(train), len(test))
print(test["created_at"].dt.strftime("%Y-%m").value_counts().sort_index())
print("train rate", round(train["escalated_72h"].mean(), 3), "test rate", round(test["escalated_72h"].mean(), 3))
print("train tickets from June:", (train["created_at"] >= "2026-06-01").sum())

> **Check.** You should see `1095 274`, test tickets from every month (42 to 56 each), `train rate 0.14 test rate 0.128`, and `train tickets from June: 169`.

The two rates are close. Nothing in this split tells you that the escalation rate changed in May.

## 5. Worked example: `split.py`

Larkfield will train the model on past tickets and use it on new ones. So Mei splits by time:

- **train:** created before 2026-05-01 (January to April),
- **valid:** May,
- **test:** 1 to 27 June. June 28 to 30 is not in the table, because those targets are not final yet.

The policy change falls between train and validation on purpose. The validation score then tells the truth: how well does a model from the old months work in the new ones? The [lesson page](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/#worked-example-splitpy) explains each important line.

Run the next cell. It saves the script as `scripts/split.py`.

In [ ]:
%%writefile scripts/split.py
"""Split the modelling table by time: train on the past, test on the future."""

import json
from pathlib import Path

import pandas as pd

IN = Path("data/model/tickets_model.csv")
OUT = Path("data/model")
VALID_FROM = pd.Timestamp("2026-05-01")
TEST_FROM = pd.Timestamp("2026-06-01")


def main():
    df = pd.read_csv(IN, parse_dates=["created_at"])
    df["split"] = "train"
    df.loc[df["created_at"] >= VALID_FROM, "split"] = "valid"
    df.loc[df["created_at"] >= TEST_FROM, "split"] = "test"
    manifest = {"rule": f"train < {VALID_FROM.date()} <= valid < {TEST_FROM.date()} <= test", "parts": {}}
    for name in ["train", "valid", "test"]:
        part = df[df["split"] == name].drop(columns="split")
        part.to_csv(OUT / f"{name}.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")
        manifest["parts"][name] = {
            "rows": len(part), "escalated": int(part["escalated_72h"].sum()),
            "escalated_rate": round(part["escalated_72h"].mean(), 3),
            "first_created": str(part["created_at"].min()), "last_created": str(part["created_at"].max()),
        }
    train_customers = set(df.loc[df["split"] == "train", "customer_id"].dropna())
    test_customers = set(df.loc[df["split"] == "test", "customer_id"].dropna())
    manifest["test_customers_also_in_train"] = len(test_customers & train_customers)
    manifest["test_customers"] = len(test_customers)
    (OUT / "split_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(manifest, indent=2))


if __name__ == "__main__":
    main()

Run the script. `%run` runs a file, like `python scripts/split.py` in a terminal.

> **Predict.** Which part will have the highest escalation rate?

In [ ]:
%run scripts/split.py

> **Check.** You should see 927 rows in train (rate 0.117), 227 in valid (0.198) and 215 in test (0.163). The last train ticket is from `2026-04-30 23:54:47`, and the first valid ticket from `2026-05-01 11:14:19`. At the end: `"test_customers_also_in_train": 77` and `"test_customers": 89`.

The script writes `train.csv`, `valid.csv`, `test.csv` and a **split manifest**, `split_manifest.json`. The manifest records the rule, and the rows, escalations and dates of each part. Priya, the data scientist, can check the split without running code.

## 6. Guided practice: check the split

Run the cell. It lists the files in `data/model`.

In [ ]:
sorted(p.name for p in Path("data/model").iterdir())

> **Check.** You should see `build_summary.json`, `split_manifest.json`, `test.csv`, `tickets_model.csv`, `train.csv` and `valid.csv`.

> **Your turn.** Read the manifest, and reconcile it with the whole table. Compute three values from `manifest`:
>
> 1. `total_rows`: the rows of the three parts, added up.
> 2. `total_escalated`: the escalations of the three parts, added up.
> 3. `dates_in_order`: `True` if the last train date is before the first validation date.
>
> Then run the check cell. The totals must be 1,369 rows and 188 escalations, as in `tickets_model.csv`.

In [ ]:
manifest = json.loads(Path("data/model/split_manifest.json").read_text())
parts = manifest["parts"]
total_rows = 0          # replace 0 with your code
total_escalated = 0     # replace 0 with your code
dates_in_order = False  # replace False with your code
print(total_rows, total_escalated, dates_in_order)

In [ ]:
expect("total_rows", total_rows, 1369)
expect("total_escalated", total_escalated, 188)
expect("dates_in_order", dates_in_order, True)

The dates are text in the format `YYYY-MM-DD HH:MM:SS`, so comparing them as text gives the same order as comparing them as times.

## 7. When to split by customer

77 of the 89 test customers also have tickets in train. Run the cell. It finds the test tickets whose customer is **not** in train.

In [ ]:
train = pd.read_csv("data/model/train.csv")
test = pd.read_csv("data/model/test.csv")
known = test["customer_id"].isin(train["customer_id"])
new = test[~known & test["customer_id"].notna()]
print("new customers:", new["customer_id"].nunique(), "with", len(new), "test tickets")
print("guest test tickets:", test["customer_id"].isna().sum())

> **Check.** You should see `new customers: 12 with 13 test tickets` and `guest test tickets: 1`.

So this test mostly measures how the model works for customers it already knows. For Larkfield, that is fine: most tickets come from customers who wrote before. A group split is needed when the model's real work is mostly about new groups, for example a model for another shop's customers.

On the lesson page, Priya trains the same simple model with the three kinds of split. The time split gives the lowest score, ROC AUC 0.673, and that is the honest number. You train that model in the next lesson.

## 8. Your turn: move the dates back one month

> **Predict.** Set `VALID_FROM` to 2026-04-01 and `TEST_FROM` to 2026-05-01. Where is the policy change now? What does the validation rate tell you? Write it down.

> **Your turn.** In the next cell, change the two dates in `scripts/split.py`:
>
> 1. Replace the line `VALID_FROM = pd.Timestamp("2026-05-01")` with the same line for `2026-04-01`.
> 2. Replace the line `TEST_FROM = pd.Timestamp("2026-06-01")` with the same line for `2026-05-01`.
>
> Use `script.replace(old, new)` once for each line. Then run the cell and the check cell.

In [ ]:
script = Path("scripts/split.py").read_text()
moved = script  # add .replace(...) for each line
Path("scripts/split.py").write_text(moved)
%run scripts/split.py

In [ ]:
manifest = json.loads(Path("data/model/split_manifest.json").read_text())
rows = [(name, part["rows"], part["escalated"]) for name, part in manifest["parts"].items()]
if manifest["parts"]["train"]["rows"] == 927:
    print("Not yet: train still has 927 rows. Change the two dates in the script.")
elif fingerprint(rows) == "0f497af079d13623":
    print("Check passed.")
else:
    print("Not yet: the parts are different. Check both dates: valid from 2026-04-01, test from 2026-05-01.")

Now the policy change is between **validation and test**. The validation rate (0.115) looks like the train rate (0.117), so every check on validation says "all is stable". The change appears only in the test set (0.181), after all the decisions are made. Put the most recent, most realistic period in validation and test.

Run the next cell. It sets the dates back and runs the script again, so the files are correct for the next lesson.

In [ ]:
script = Path("scripts/split.py").read_text()
script = (script
    .replace('VALID_FROM = pd.Timestamp("2026-04-01")', 'VALID_FROM = pd.Timestamp("2026-05-01")')
    .replace('TEST_FROM = pd.Timestamp("2026-05-01")', 'TEST_FROM = pd.Timestamp("2026-06-01")'))
Path("scripts/split.py").write_text(script)
%run scripts/split.py

> **Check.** You should see 927, 227 and 215 rows again.

## 9. Failure case: one random split is one draw

Tomás splits the 1,369 rows at random, gets a high score, and reports that one number. But a random split is one draw. Run the cell. It makes 20 random 20% test sets and prints the test escalation rate of each.

> **Predict.** How far apart will the lowest and the highest rate be?

In [ ]:
rates = [round(float(model.sample(frac=0.2, random_state=seed)["escalated_72h"].mean()), 3) for seed in range(20)]
print(rates)
print("lowest", min(rates), "highest", max(rates))

> **Check.** You should see `lowest 0.102 highest 0.161`. No draw shows the May change: every random test set mixes all six months.

**Cause:** different tickets land in test with each draw, and a random split lets the model learn from the same months that it is tested on. On the lesson page, the same model scores from 0.665 to 0.827 across 20 random draws.

**Fix:** split by time, as `split.py` does, so the validation score copies real use. If you also use random splits, report the spread across draws, not one number. Write the rule and the counts in `split_manifest.json`, so a reader can see which test the score comes from.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that the split must copy real use. Split by time when the model predicts the future, by group when it must work for new customers, and at random only for independent rows. On the real data, the escalation rate rises from 11.7% (train) to 19.8% (May) and 16.3% (June). A time split shows the change; a random split hides it. `split.py` writes `train.csv`, `valid.csv`, `test.csv` and `split_manifest.json`.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Fit transformations correctly](https://learning.nextia-ai.com/courses/sql/m05/fit-transformations-correctly/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.